# Extractor de Tablas — Presupuesto Participativo CDMX

Este notebook permite explorar y probar la extracción de datos desde fotografías de tablas del Presupuesto Participativo de la Ciudad de México.

**Flujo:**
1. Configurar API key de Anthropic
2. Cargar una imagen de ejemplo
3. Extraer datos con Claude Vision
4. Validar colonias contra catálogo oficial
5. Exportar resultados a CSV


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Image as IPImage

# Asegurarse de que el módulo src sea importable
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.extractor import TableExtractor

print("Módulos cargados correctamente.")

## 1. Configuración

Asegúrate de tener tu API key de Anthropic configurada. Puedes crear un archivo `.env` en la raíz del proyecto:

```
ANTHROPIC_API_KEY=sk-ant-...
```

O configurarla directamente:

In [ ]:
# Opcional: si no tienes .env, descomenta y agrega tu key aquí
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

extractor = TableExtractor()
print("Extractor inicializado.")

## 2. Explorar imágenes de ejemplo

Coloca tus imágenes en `images/examples/` y ejecuta la celda siguiente para listarlas.

In [ ]:
IMAGES_DIR = ROOT / "images" / "examples"
extensions = {".jpg", ".jpeg", ".png", ".webp"}

imagenes = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in extensions]

if imagenes:
    print(f"Imágenes encontradas ({len(imagenes)}):")
    for img in sorted(imagenes):
        print(f"  {img.name}")
else:
    print("No hay imágenes en images/examples/")
    print("Agrega fotografías de tablas del PP para continuar.")

## 3. Previsualizar una imagen

In [ ]:
# Cambia el nombre de la imagen que quieres probar
# Imágenes de ejemplo disponibles:
#   - "iztapalapa 2015.png"
#   - "Miguel Hidalgo - 2015.png"
#   - "Magdalena Contreras - 2018.png"
#   - "Tláhuac 2015.png"
#   - "tlalpan 2015.png"
IMAGEN_PRUEBA = "iztapalapa 2015.png"  # <-- EDITA ESTE NOMBRE

img_path = IMAGES_DIR / IMAGEN_PRUEBA

if img_path.exists():
    display(IPImage(filename=str(img_path), width=700))
else:
    print(f"Imagen no encontrada: {img_path}")
    print("Verifica que el nombre coincida con algún archivo en images/examples/")

## 4. Extraer datos de la tabla

In [ ]:
if img_path.exists():
    print(f"Extrayendo datos de: {IMAGEN_PRUEBA}")
    df = extractor.extract(str(img_path))
    
    print(f"\n=== Metadatos del documento ===")
    print(f"  Alcaldía             : {df.attrs.get('alcaldia', 'No detectada')}")
    print(f"  Unidad Responsable   : {df.attrs.get('unidad_responsable', 'No detectada')}")
    print(f"  Año                  : {df.attrs.get('anio', 'No detectado')}")
    print(f"  Página               : {df.attrs.get('pagina', 'No detectada')}")
    print(f"  Notas                : {df.attrs.get('notas', 'Ninguna')}")
    
    print(f"\n=== Datos extraídos ({len(df)} filas × {len(df.columns)} columnas) ===")
    display(df)
else:
    print("Agrega una imagen de prueba primero.")

## 5. Validar colonia contra catálogo oficial

In [ ]:
# Cargar catálogo de colonias
colonias_path = ROOT / "data" / "colonias" / "colonias_cdmx.csv"
colonias_df = pd.read_csv(colonias_path)

print(f"Catálogo cargado: {len(colonias_df)} colonias en {colonias_df['alcaldia'].nunique()} alcaldías\n")
print(colonias_df.groupby("alcaldia").size().to_string())

In [ ]:
# Validar una colonia específica del catálogo
# Las colonias están en la columna "COLONIA O PUEBLO ORIGINARIO" de la tabla extraída

# Si ya extrajiste datos (celda anterior), muestra las colonias únicas:
if 'df' in dir() and not df.empty:
    col_colonia = next(
        (c for c in df.columns if "colonia" in c.lower() or "pueblo" in c.lower()),
        None
    )
    if col_colonia:
        colonias_en_tabla = df[col_colonia].dropna().unique()
        print(f"Colonias en la tabla extraída ({len(colonias_en_tabla)}):")
        for c in colonias_en_tabla:
            print(f"  - {c}")

# Validar una colonia contra el catálogo oficial
BUSCAR_COLONIA = "Tepito"   # <-- CAMBIA A UNA COLONIA DE LA TABLA
BUSCAR_ALCALDIA = None      # Opcional: "CUAUHTÉMOC" para filtrar por alcaldía

resultado = extractor.validate_colonia(BUSCAR_COLONIA, BUSCAR_ALCALDIA)
print(f"\nValidación de '{BUSCAR_COLONIA}' en catálogo CDMX:")
print(f"  Encontrada: {resultado['encontrado']}")
for c in resultado.get('coincidencias', []):
    print(f"  - {c['colonia']} / {c['alcaldia']}")

## 6. Procesar múltiples imágenes en lote

In [ ]:
# Procesar todas las imágenes de ejemplo en lote
# Genera un CSV por imagen + data/processed/consolidado.csv
df_consolidado = extractor.process_directory(
    images_dir="images/examples",
    output_dir="data/processed"
)
print(f"\nTotal de filas: {len(df_consolidado)}")
display(df_consolidado.head(20))

## 7. Guardar resultados

In [ ]:
if 'df' in dir() and not df.empty:
    output_path = ROOT / "data" / "processed" / IMAGEN_PRUEBA.replace(".jpg", ".csv").replace(".png", ".csv")
    extractor.save_csv(df, str(output_path))
    print(f"Guardado en: {output_path}")
else:
    print("No hay datos que guardar todavía.")